<a href="https://colab.research.google.com/github/aljubic1/bioinformatika_projekt_alj/blob/main/notebooks/03_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. Modeling

## Cilj

Cilj ovog notebooka je trenirati modele strojnog učenja za klasifikaciju peptida na AMP i non-AMP klase. Koristit će se Random Forest i Support Vector Machine (SVM), a modeli će se usporediti na temelju osnovnih klasifikacijskih metrika.


In [1]:
# Uvoz pandas biblioteke za rad s tabličnim podacima.
import pandas as pd

# Uvoz funkcije za podjelu podataka na trening i testni skup.
from sklearn.model_selection import train_test_split

# Uvoz StandardScaler-a za skaliranje značajki.
from sklearn.preprocessing import StandardScaler

# Uvoz Random Forest klasifikatora.
from sklearn.ensemble import RandomForestClassifier

# Uvoz SVM klasifikatora.
from sklearn.svm import SVC

# Uvoz metrika za evaluaciju.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# Putanja do skupa značajki.
features_path = "features.csv"

# Učitavanje skupa značajki.
features_df = pd.read_csv(features_path)

# Prikaz prvih pet redaka.
features_df.head()

,ID,SEQUENCE,length,net_charge,hydrophobicity,label,AAC_A,AAC_C,AAC_D,AAC_E,...,AAC_M,AAC_N,AAC_P,AAC_Q,AAC_R,AAC_S,AAC_T,AAC_V,AAC_W,AAC_Y
0,1390,CFQWQRNARKVR,12,4,-1.458333,1,0.083333,0.083333,0.000000,0.000000,...,0.0,0.083333,0.0,0.166667,0.250000,0.000000,0.000000,0.083333,0.083333,0.000000
1,Seq8055_sampling_method=Wang-et-al_AMP=0_rep2,TEETKVIDSRLVSDGYQ,17,-2,-0.817647,0,0.000000,0.000000,0.117647,0.117647,...,0.0,0.000000,0.0,0.058824,0.058824,0.117647,0.117647,0.117647,0.000000,0.058824
2,1661,WLNALLHHGLNCAKGVL,17,3,0.605882,1,0.117647,0.058824,0.000000,0.000000,...,0.0,0.117647,0.0,0.000000,0.000000,0.000000,0.000000,0.058824,0.058824,0.000000
3,1552,RLWRIVVIRVAR,12,4,0.691667,1,0.083333,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.000000,0.333333,0.000000,0.000000,0.250000,0.083333,0.000000
4,1585,ILGTILGLLKSL,12,1,1.816667,1,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.000000,0.000000,0.083333,0.083333,0.000000,0.000000,0.000000


## Pregled skupa značajki

Prije treniranja modela provjerava se broj uzoraka i broj značajki kako bi se potvrdilo da je skup podataka ispravno učitan.

In [3]:
# Dimenzije skupa značajki.
features_df.shape

(3344, 26)

## Priprema podataka

Za treniranje modela potrebno je odvojiti ulazne značajke (X) od ciljne varijable (y). Stupci `ID` i `SEQUENCE` ne koriste se za treniranje jer nisu numeričke značajke.

In [4]:
# Ulazne značajke.
X = features_df.drop(columns=["ID", "SEQUENCE", "label"])

# Ciljna varijabla.
y = features_df["label"]

print("Dimenzije X:", X.shape)
print("Dimenzije y:", y.shape)

Dimenzije X: (3344, 23)
Dimenzije y: (3344,)


## Podjela na trening i testni skup

Skup podataka dijeli se na trening (80 %) i testni skup (20 %). Trening skup koristi se za učenje modela, dok se testni skup koristi za procjenu uspješnosti modela na prethodno neviđenim podacima.

In [5]:
# Podjela na trening i testni skup.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (2675, 23)
X_test : (669, 23)
y_train: (2675,)
y_test : (669,)


Skaliranje podataka

Random Forest ne zahtijeva skaliranje, ali ga zahtijeva SVM. Kako uspoređujemo oba modela, napravit ćemo skaliranje jednom.
## Standardizacija značajki

Za potrebe SVM modela numeričke značajke standardiziraju se primjenom StandardScaler algoritma. Skaliranje se provodi koristeći samo trening skup kako bi se izbjeglo curenje informacija (data leakage).

In [6]:
# Kreiranje StandardScaler objekta.
scaler = StandardScaler()

# Skaliranje trening skupa.
X_train_scaled = scaler.fit_transform(X_train)

# Skaliranje testnog skupa.
X_test_scaled = scaler.transform(X_test)

Zašto radimo ovim redoslijedom?

Ovaj redoslijed ( učitaj → pripremi X i y → podijeli → skaliraj → treniraj) je standardna praksa u strojnom učenju i često se koristi u znanstvenim radovima. Tako će i tvoj projekt biti metodološki ispravan.

Kad ovo završiš, krećemo na:
🌳 Slučajna šuma
🎯 SVM
📊 Usporedbu njihovih rezultata kroz točnost, preciznost, odziv i F1 mjeru.

#slucajna suma - smanjenje
## Treniranje Random Forest modela

Random Forest je ansambl metoda strojnog učenja koja koristi veći broj stabala odlučivanja. Konačna klasifikacija dobiva se glasanjem svih stabala, čime se povećava stabilnost modela i smanjuje mogućnost prenaučenosti (overfitting).

In [7]:
# Kreiranje Random Forest modela.
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Treniranje modela.
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [9]:
#predvidanje
# Predikcija na testnom skupu.
rf_predictions = rf_model.predict(X_test)

In [10]:
#osnove metrike
# Izračun osnovnih metrika.
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)

print("=" * 40)
print("Random Forest")
print("=" * 40)
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-score : {rf_f1:.4f}")
print("=" * 40)

Random Forest
Accuracy : 0.9312
Precision: 0.9235
Recall   : 0.9401
F1-score : 0.9318


Rezultati koje si dobila:

Metrika	Vrijednost
Točnost	0,9312
Preciznost	0,9235
Podsjetiti	0,9401
F1-rezultat	0,9318
Što to znači?
Točnost = 93,12 % → model je ispravno klasificirao oko 93 % peptida.
Preci→ kada model kaže da je peptid AMP, u većini slučajeva je u pravu.
Recall = 94.01 % → model uspješno pronalazi veliki dio stvarnih AMP peptida.
F1-score = 93.18 % → vrlo dobra ravnoteža između preciznosti i prisjećanja.

To su vrlo dobri rezultati za ovakav projekt.

### Tumačenje rezultata Random Forest modela

Random Forest model ostvario je vrlo dobre rezultate na testnom skupu podataka. Točnost klasifikacije iznosi **93,12 %**, što pokazuje da model uspješno razlikuje antimikrobne i neantimikrobne peptide.

Posebno se ističe vrijednost odziva (Recall) od **94,01 %**, što znači da model prepoznaje velik broj stvarnih antimikrobnih peptida. F1-mjera od **93,18 %** potvrđuje dobru ravnotežu između preciznosti i odziva te ukazuje na stabilne performanse modela.

Sada radimo isti postupak za SVM :

Treniranje SVM modela
Predviđanje
Izračun metrike
Kratko tumačenje rezultata

Tek nakon toga napravit ćemo tablicu usporedbe oba modela.

Idemo na SVM (Support Vector Machine) . Radit ćemo istim redoslijedom kao i za Random Forest kako bi notebook bio uredan i lako usporediv.

## Treniranje SVM modela

Support Vector Machine (SVM) je algoritam strojnog učenja koji pronalazi optimalnu granicu razdvajanja između klasa. Za razliku od Random Forest modela, SVM zahtijeva standardizirane ulazne značajke, zbog čega se koristi prethodno skalirani trening i testni skup.

In [11]:
#treniranje modela
# Kreiranje SVM modela.
svm_model = SVC(
    kernel="rbf",
    random_state=42
)

# Treniranje modela.
svm_model.fit(X_train_scaled, y_train)

SVC(random_state=42)

In [12]:
#predvidanje
# Predikcija na testnom skupu.
svm_predictions = svm_model.predict(X_test_scaled)

In [13]:
#izracun metrike
# Izračun osnovnih metrika.
svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_precision = precision_score(y_test, svm_predictions)
svm_recall = recall_score(y_test, svm_predictions)
svm_f1 = f1_score(y_test, svm_predictions)

print("=" * 40)
print("Support Vector Machine")
print("=" * 40)
print(f"Accuracy : {svm_accuracy:.4f}")
print(f"Precision: {svm_precision:.4f}")
print(f"Recall   : {svm_recall:.4f}")
print(f"F1-score : {svm_f1:.4f}")
print("=" * 40)

Support Vector Machine
Accuracy : 0.9297
Precision: 0.9309
Recall   : 0.9281
F1-score : 0.9295


### Tumačenje rezultata SVM modela

SVM model ostvario je vrlo dobre rezultate na testnom skupu podataka. Točnost klasifikacije iznosi **92,97 %**, što pokazuje da model uspješno razlikuje antimikrobne i neantimikrobne peptide.

Preciznost modela iznosi **93,09 %**, što znači da je većina peptida koje je model klasificirao kao antimikrobne doista pripadala toj klasi. Odziv (Recall) od **92,81 %** pokazuje da model uspješno prepoznaje velik broj stvarnih antimikrobnih peptida.

F1-mjera od **92,95 %** potvrđuje dobru ravnotežu između preciznosti i odziva te ukazuje na stabilne performanse SVM modela.

In [14]:
# Usporedba rezultata modela.
comparison = pd.DataFrame({
    "Model": ["Random Forest", "SVM"],
    "Accuracy": [rf_accuracy, svm_accuracy], #Točnost
    "Precision": [rf_precision, svm_precision], #Preciznost
    "Recall": [rf_recall, svm_recall],
    "F1-score": [rf_f1, svm_f1]
})

comparison

,Model,Accuracy,Precision,Recall,F1-score
0,Random Forest,0.931241,0.923529,0.940120,0.931751
1,SVM,0.929746,0.930931,0.928144,0.929535
